In [0]:
dbutils.widgets.text("catalog", "bank_risk_dev")
dbutils.widgets.dropdown("step", "all", ["all", "functions", "grants"])
catalog = dbutils.widgets.get("catalog")
step = dbutils.widgets.get("step")

In [0]:
if step in ("all", "functions"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.governance")

    spark.sql(f"""
    CREATE OR REPLACE FUNCTION {catalog}.governance.mask_pii(v STRING)
    RETURNS STRING
    COMMENT 'Muestra el valor completo solo a risk_team; a los demás les deja ver los últimos 4 caracteres'
    RETURN CASE
      WHEN v IS NULL THEN NULL
      WHEN is_account_group_member('risk_team') THEN v
      ELSE concat(repeat('*', greatest(length(v) - 4, 0)), right(v, 4))
    END
    """)

    spark.sql(f"""
    CREATE OR REPLACE FUNCTION {catalog}.governance.mask_name(v STRING)
    RETURNS STRING
    COMMENT 'Muestra el nombre completo solo a risk_team; a los demás solo la inicial'
    RETURN CASE
      WHEN v IS NULL THEN NULL
      WHEN is_account_group_member('risk_team') THEN v
      ELSE concat(substr(v, 1, 1), '*****')
    END
    """)

    spark.sql(f"""
    CREATE OR REPLACE FUNCTION {catalog}.governance.alert_filter(rule_code STRING)
    RETURNS BOOLEAN
    COMMENT 'risk_team ve todas las alertas; fraud_analysts solo las de reglas de fraude. Las alertas PLD son confidenciales'
    RETURN is_account_group_member('risk_team')
      OR (is_account_group_member('fraud_analysts')
          AND rule_code IN ('R02_BENEF_DISPOSITIVO_NUEVO', 'R03_NOCTURNO'))
    """)

In [0]:
if step in ("all", "grants"):
    access = {
        "credit_analysts": ["dim_customer", "dim_loan", "dim_date", "fact_loan_month", "mv_credit_risk"],
        "fraud_analysts": ["dim_customer", "dim_account", "dim_bank", "dim_rule", "dim_date",
                           "fact_transfer", "fact_alert", "mv_fraud", "mv_alerts"],
    }

    for group in ["credit_analysts", "fraud_analysts", "risk_team"]:
        spark.sql(f"GRANT USE CATALOG ON CATALOG {catalog} TO `{group}`")
        spark.sql(f"GRANT USE SCHEMA ON SCHEMA {catalog}.gold TO `{group}`")

    spark.sql(f"GRANT SELECT ON SCHEMA {catalog}.gold TO `risk_team`")

    for group, tables in access.items():
        for t in tables:
            spark.sql(f"GRANT SELECT ON TABLE {catalog}.gold.{t} TO `{group}`")

In [0]:
if step in ("all", "grants"):
    display(spark.sql(f"""
    SELECT table_name, column_name, mask_name
    FROM {catalog}.information_schema.column_masks
    WHERE table_schema = 'gold'
    """))

    display(spark.sql(f"""
    SELECT table_name, filter_name, target_columns
    FROM {catalog}.information_schema.row_filters
    WHERE table_schema = 'gold'
    """))